# Make AI clips for your Amazon ad agent (free GPU)

**Status: UNTESTED.** It was written without access to a GPU. Expect to adjust something on the first run, and start with `MAX_SCENES = 2`.

**What it does:** reads `projects/<ASIN>/scenes.md` from your GitHub repo, makes one ~5-second clip per scene with an open-source video model, and saves them back into `projects/<ASIN>/assets/` as `ai_s01.mp4`, `ai_s02.mp4` ... The next *render* in GitHub picks them up automatically, adds an "AI-generated scene" tag, and adds the AI note to the description.

**Before you start**
1. In GitHub, run **Make video** with **mode: scenes**. Read and edit `scenes.md` (keep scenes faceless, no product claims).
2. Create a GitHub *fine-grained token*: GitHub, Settings, Developer settings, Personal access tokens, Fine-grained. Repository access: **only** `amazon-video-agent`. Permission: **Contents: Read and write**. Short expiry (30 days).
3. **Kaggle:** new notebook, File, Import Notebook (upload this file). Settings: Accelerator = GPU (T4 or P100), Internet = On (needs phone verification). Add-ons, Secrets: add `GITHUB_TOKEN`.
   **Colab:** Runtime, Change runtime type, T4 GPU. Click the key icon, add secret `GITHUB_TOKEN`, enable notebook access.
4. Never paste the token into a notebook cell, a chat, or the repo.
5. Edit the settings cell below (at least `ASIN`), then **Run all**.

**Good to know:** free GPU time is limited and a clip can take several minutes. Results are saved as they finish, so you can stop and re-run; finished clips are skipped. Check every clip: AI video can bend hands and objects. Delete bad ones from `assets/` on GitHub. Realistic AI scenes must be labelled in YouTube's upload form.

In [ ]:
# ---- settings: edit these -------------------------------------------------
ASIN = "B0DN5N1FL5"                 # the product folder name in projects/
REPO = "Swar-i-K/amazon-video-agent"
BRANCH = "main"
MODEL = "Lightricks/LTX-Video"      # open-source video model in diffusers format
WIDTH, HEIGHT = 704, 480            # multiples of 32
FRAMES, FPS = 121, 24               # frames must be 8*k+1 (121 = about 5 seconds)
STEPS = 30                          # fewer = faster, lower quality
MAX_SCENES = 2                      # start small; raise to 10-12 once it works
SEED = 7
PUSH_TO_GITHUB = True               # False = only make a zip to download
SUFFIX = " No text, no logos, no watermarks, no visible faces."
NEGATIVE = "worst quality, blurry, jittery, distorted, deformed, text, watermark, logo, face, extra fingers"

In [ ]:
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg requests protobuf

In [ ]:
import base64, gc, os, re, shutil
import requests

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
if not TOKEN:
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
TOKEN = TOKEN or os.environ.get("GITHUB_TOKEN")
assert TOKEN, "Add a GITHUB_TOKEN secret (see the instructions at the top)."

HEAD = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
API = f"https://api.github.com/repos/{REPO}/contents"


def gh_raw(path):
    r = requests.get(f"{API}/{path}", params={"ref": BRANCH}, headers={**HEAD, "Accept": "application/vnd.github.raw+json"}, timeout=120)
    r.raise_for_status()
    return r


scenes_md = gh_raw(f"projects/{ASIN}/scenes.md").text
scenes = []
for line in scenes_md.splitlines():
    m = re.match(r"^\[(\w+)\]\s*type:\s*(\w+)\s*\|\s*ref:\s*([^|]+?)\s*\|\s*prompt:\s*(.+)$", line.strip())
    if m:
        scenes.append({"id": m.group(1), "type": m.group(2).lower(), "ref": m.group(3).strip(), "prompt": m.group(4).strip()})
scenes = scenes[:MAX_SCENES]
print(f"{len(scenes)} scene(s) to make:")
for s in scenes:
    print(" ", s["id"], s["type"], "|", s["prompt"][:90])

In [ ]:
import io
import torch
from PIL import Image
from diffusers import LTXImageToVideoPipeline, LTXPipeline
from diffusers.utils import export_to_video

assert torch.cuda.is_available(), "No GPU. Turn on a GPU accelerator in the notebook settings."
major = torch.cuda.get_device_capability()[0]
DTYPE = torch.bfloat16 if major >= 8 else torch.float16     # older GPUs (T4/P100) lack bfloat16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)

OUT = "clips"
os.makedirs(OUT, exist_ok=True)
_pipe = {"kind": None, "obj": None}


def load(kind):
    """Keep only one pipeline in memory (free GPUs have little room)."""
    if _pipe["kind"] == kind:
        return _pipe["obj"]
    _pipe["obj"] = None
    gc.collect()
    torch.cuda.empty_cache()
    cls = LTXPipeline if kind == "t2v" else LTXImageToVideoPipeline
    pipe = cls.from_pretrained(MODEL, torch_dtype=DTYPE)
    pipe.vae.enable_tiling()
    pipe.enable_model_cpu_offload()
    _pipe.update(kind=kind, obj=pipe)
    return pipe


def fit_image(img):
    """Place the product photo on a plain canvas of the video size without cropping the product."""
    img = img.convert("RGB")
    corner = img.getpixel((2, 2))
    s = min(WIDTH / img.width, HEIGHT / img.height)
    img = img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))))
    canvas = Image.new("RGB", (WIDTH, HEIGHT), corner)
    canvas.paste(img, ((WIDTH - img.width) // 2, (HEIGHT - img.height) // 2))
    return canvas


def make(scene):
    path = f"{OUT}/ai_{scene['id']}.mp4"
    if os.path.exists(path):
        print("  skip (already made):", path)
        return path
    gen = torch.Generator("cpu").manual_seed(SEED)
    kw = dict(prompt=scene["prompt"] + SUFFIX, negative_prompt=NEGATIVE, width=WIDTH, height=HEIGHT,
              num_frames=FRAMES, num_inference_steps=STEPS, generator=gen)
    if scene["type"] == "product" and scene["ref"].lower() != "none":
        pipe = load("i2v")
        ref = Image.open(io.BytesIO(gh_raw(f"projects/{ASIN}/assets/{scene['ref']}").content))
        frames = pipe(image=fit_image(ref), **kw).frames[0]
    else:
        pipe = load("t2v")
        frames = pipe(**kw).frames[0]
    export_to_video(frames, path, fps=FPS)
    return path


made = []
# mood scenes first (one model load), then product scenes (second model load)
for s in sorted(scenes, key=lambda x: x["type"] == "product"):
    print("making", s["id"], "...")
    try:
        made.append(make(s))
        print("  done:", made[-1])
    except Exception as e:
        print("  FAILED:", repr(e)[:300])
print("clips made:", made)

In [ ]:
def push(local, remote):
    url = f"{API}/{remote}"
    r = requests.get(url, headers=HEAD, params={"ref": BRANCH}, timeout=60)
    body = {"message": f"AI clip {os.path.basename(local)}", "branch": BRANCH,
            "content": base64.b64encode(open(local, "rb").read()).decode()}
    if r.status_code == 200:
        body["sha"] = r.json()["sha"]
    requests.put(url, headers=HEAD, json=body, timeout=300).raise_for_status()


files = sorted(f for f in os.listdir(OUT) if f.endswith(".mp4"))
if PUSH_TO_GITHUB:
    for f in files:
        push(f"{OUT}/{f}", f"projects/{ASIN}/assets/{f}")
        print("uploaded", f)
shutil.make_archive("ai_clips", "zip", OUT)
print("Also saved ai_clips.zip (download it from the notebook's output files if you prefer to upload by hand).")
print("Next: in GitHub, delete any clip that looks wrong, then run Make video with mode render.")